In [2]:
from itertools import product
import numpy as np
import random
import os
import shutil

In [3]:
def generate_components(gym_type):
    if gym_type == 'Transformer':
        model_names = ['_'.join(list(_)) for _ in list(product(['TSGym'],
                                                            ['False', 'True'], # gym_x_mark
                                                            ['False', 'True'], # gym_series_sampling
                                                            ['None', 'Stat', 'RevIN', 'DishTS'], # gym_series_norm
                                                            ['None', 'MA', 'MoEMA', 'DFT'], # gym_series_decomp
                                                            ['False', 'True'], # gym_channel_independent
                                                            ['inverted-encoding', 'series-encoding', 'series-patching'], # gym_input_embed
                                                            ['Transformer'], # gym_network_architecture
                                                            ['self-attention', 'auto-correlation', 'sparse-attention', 'frequency-enhanced-attention', 'destationary-attention'], # gym_attn
                                                            ['null', 'self-attention', 'sparse-attention', 'frequency-enhanced-attention'], # gym_feature_attn， null
                                                            ['True'], # gym_encoder_only
                                                            ['False'], # gym_frozen
                                                            ['HP'],
                                                            ['48', '96', '192', '512'], # sequence length ['48', '96', '192', '512']
                                                            ['64-256', '256-1024'], # d_model, d_ff
                                                            ['2', '3'], # encoder layers
                                                            ['10', '20', '50'], # training epochs
                                                            ['MAE','MSE','HUBER'], # loss functions ['MSE', 'MAE', 'HUBER']
                                                            ['0.0001', '0.001'], # learning rate
                                                            ['null', 'type1'] # learning rate strategy
                                                            ))]
    elif gym_type == 'LLM':
        model_names = ['_'.join(list(_)) for _ in list(product(['TSGym'],
                                                            ['False', 'True'], # gym_x_mark
                                                            ['False'], # gym_series_sampling
                                                            ['None', 'Stat', 'RevIN', 'DishTS'], # gym_series_norm
                                                            ['None', 'MA', 'MoEMA', 'DFT'], # gym_series_decomp
                                                            ['True'], # gym_channel_independent
                                                            ['series-patching'], # gym_input_embed
                                                            ['LLM-GPT4TS', 'LLM-TimeLLM'], # gym_network_architecture
                                                            ['self-attention'], # gym_attn
                                                            ['null'], # gym_feature_attn
                                                            ['True'], # gym_encoder_only
                                                            ['False', 'True'], # gym_frozen
                                                            ['HP'],
                                                            ['48', '96', '192', '512'], # sequence length 
                                                            ['64-256', '256-1024'], # d_model, d_ff
                                                            ['6'], # encoder layers
                                                            ['10', '20', '50'], # training epochs
                                                            ['MAE','MSE', 'HUBER'], # loss functions
                                                            ['0.0001', '0.001'], # learning rate
                                                            ['null', 'type1'] # learning rate strategy
                                                            ))]
    elif gym_type == 'TSFM':
        model_names = ['_'.join(list(_)) for _ in list(product(['TSGym'],
                                                            ['False', 'True'], # gym_x_mark
                                                            ['False'], # gym_series_sampling
                                                            ['None', 'Stat', 'RevIN', 'DishTS'], # gym_series_norm
                                                            ['None', 'MA', 'MoEMA', 'DFT'], # gym_series_decomp
                                                            ['True'], # gym_channel_independent
                                                            ['series-patching'], # gym_input_embed
                                                            ['TSFM-Timer', 'TSFM-Moment'], # gym_network_architecture
                                                            ['self-attention'], # gym_attn
                                                            ['null'], # gym_feature_attn
                                                            ['True'], # gym_encoder_only
                                                            ['False', 'True'], # gym_frozen
                                                            ['HP'],
                                                            ['48', '96', '192', '512'], # sequence length 
                                                            ['64-256', '256-1024'], # d_model, d_ff
                                                            ['6'], # encoder layers
                                                            ['10', '20', '50'], # training epochs
                                                            ['MAE','MSE', 'HUBER'], # loss functions
                                                            ['0.0001', '0.001'], # learning rate
                                                            ['null', 'type1'] # learning rate strategy
                                                            ))]
    else:
        model_names = ['_'.join(list(_)) for _ in list(product(['TSGym'],
                                                        ['False', 'True'], # gym_x_mark
                                                        ['False', 'True'], # gym_series_sampling
                                                        ['None', 'Stat', 'RevIN', 'DishTS'], # gym_series_norm
                                                        ['None', 'MA', 'MoEMA', 'DFT'], # gym_series_decomp
                                                        ['False', 'True'], # gym_channel_independent
                                                        ['inverted-encoding', 'series-encoding', 'series-patching'], # gym_input_embed
                                                        ['MLP', 'GRU'], # gym_network_architecture
                                                        ['null'], # gym_attn
                                                        ['null', 'self-attention', 'sparse-attention', 'frequency-enhanced-attention'], # gym_feature_attn, null
                                                        ['True'], # gym_encoder_only
                                                        ['False'], # gym_frozen
                                                        ['HP'],
                                                        ['48', '96', '192', '512'], # sequence length ['48', '96', '192']
                                                        ['64-256', '256-1024'], # d_model, d_ff
                                                        ['2', '3'], # encoder layers
                                                        ['10', '20', '50'], # training epochs
                                                        ['MAE','MSE', 'HUBER'], # loss functions
                                                        ['0.0001', '0.001'], # learning rate
                                                        ['null', 'type1'] # learning rate strategy
                                                        ))]
    # 初步删除可能存在问题的组合
    # Update 20250726 cc
    valid_model_names = []
    for model_name in model_names:
        compoents_list = model_name.split('_')
        _channel_independent = compoents_list[5]
        _input_embed = compoents_list[6]
        _series_sampling = compoents_list[2]
        _feature_attn = compoents_list[9]
        # 去掉冲突的组合
        if _channel_independent == 'True' and _input_embed == 'inverted-encoding':
            continue
        if _channel_independent == 'False' and _input_embed == 'series-patching':
            continue
        if _channel_independent == 'True' and _feature_attn != 'null':
            continue
        if _series_sampling == 'True' and _input_embed == 'inverted-encoding':
            continue
        valid_model_names.append(model_name)

    return valid_model_names

In [4]:
for gym_type in ['Transformer', 'non_Transformer']:
    random.seed(42)
    model_names = generate_components(gym_type)
    print(len(model_names))
    model_names = random.sample(model_names, 1500)
    for dataset in ['zafnoo']:
        if dataset in ['NASDAQ', 'NYSE', 'ILI']:
            pred_len_list = [24,36,48,60]
        else:
            pred_len_list = [96,192,336,720]
        for pl in pred_len_list:
            # 模板文件路径
            if 'ETT' in dataset:
                template_path = f'../scripts/long_term_forecast/{dataset}_script/TSGym_{dataset}.sh'
            else:
                template_path = f'../scripts/long_term_forecast/{dataset}_script/TSGym_pl{pl}.sh'
            
            # 输出目录
            output_dir = f'../scripts/long_term_forecast/{dataset}_script/gym_{gym_type}_pl{pl}'

            # 确保输出目录存在
            if os.path.exists(output_dir):
                print('delete current folder!')
                shutil.rmtree(output_dir)
            os.makedirs(output_dir, exist_ok=True)

            # 读取模板内容
            with open(template_path, 'r') as file:
                template_content = file.read()

            # 对于每个模型名称，生成一个 shell 脚本
            for model_name in model_names:
                file_name = model_name

                HP = model_name[model_name.find('_HP')+4:]
                model_name = model_name[:model_name.find('_HP')]

                seq_len, dm_df, el, epochs, loss, lr, lr_strategy = HP.split('_')
                dm, df = dm_df.split('-')[0], dm_df.split('-')[1]

                # 替换模型名称
                script_content = template_content.replace('$model_name', model_name)
                script_content = script_content.replace(f'$seq_len', seq_len)
                script_content = script_content.replace(f'$d_model', dm)
                script_content = script_content.replace(f'$d_ff', df)
                script_content = script_content.replace(f'$e_layers', el)
                script_content = script_content.replace(f'$train_epochs', epochs)
                script_content = script_content.replace(f'$loss', loss)
                script_content = script_content.replace(f'$learning_rate', lr)
                script_content = script_content.replace(f'$lradj', lr_strategy)
                
                # 定义输出文件名
                output_file = os.path.join(output_dir, f'{file_name}.sh')
                
                # 写入新的 shell 脚本
                with open(output_file, 'w') as file:
                    file.write(script_content)

                # print(f'Generated {output_file}')

1474560
589824


In [5]:
old_model_names_non_Transformer = [x for x in os.listdir('/data/nishome/user1/minqi/TSGym/scripts/long_term_forecast/AQShunyi_script/gym_non_Transformer_pl96') if 'TSGym' in x]
old_model_names_Transformer = [x for x in os.listdir('/data/nishome/user1/minqi/TSGym/scripts/long_term_forecast/AQShunyi_script/gym_Transformer_pl96') if 'TSGym' in x]

In [9]:
# 生成一份AQShunyi的新的script在A800上跑，要求和之前的setting不能一样
# 但其实很难重复

for gym_type in ['Transformer', 'non_Transformer']:
    random.seed(1)
    model_names = generate_components(gym_type)
    print(len(model_names))
    model_names = random.sample(model_names, 1500)
    # 去掉之前采样的
    if gym_type == 'Transformer':
        model_names = [x for x in model_names if x not in old_model_names_Transformer]
    else:
        model_names = [x for x in model_names if x not in old_model_names_non_Transformer]
    print(len(model_names))
    for dataset in ['AQShunyi']:
        if dataset in ['NASDAQ','NYSE']:
            pred_len_list = [24,36,48,60]
        else:
            pred_len_list = [96,192,336,720]
        for pl in pred_len_list:
            # 模板文件路径
            if 'ETT' in dataset:
                template_path = f'../scripts/long_term_forecast/{dataset}_script/TSGym_{dataset}.sh'
            else:
                template_path = f'../scripts/long_term_forecast/{dataset}_script/TSGym_pl{pl}.sh'
            
            # 输出目录
            output_dir = f'../scripts/long_term_forecast/{dataset}_script/gym_{gym_type}_pl{pl}'

            # 确保输出目录存在
            if os.path.exists(output_dir):
                print('delete current folder!')
                shutil.rmtree(output_dir)
            os.makedirs(output_dir, exist_ok=True)

            # 读取模板内容
            with open(template_path, 'r') as file:
                template_content = file.read()

            # 对于每个模型名称，生成一个 shell 脚本
            for model_name in model_names:
                file_name = model_name

                HP = model_name[model_name.find('_HP')+4:]
                model_name = model_name[:model_name.find('_HP')]

                seq_len, dm_df, el, epochs, loss, lr, lr_strategy = HP.split('_')
                dm, df = dm_df.split('-')[0], dm_df.split('-')[1]

                # 替换模型名称
                script_content = template_content.replace('$model_name', model_name)
                script_content = script_content.replace(f'$seq_len', seq_len)
                script_content = script_content.replace(f'$d_model', dm)
                script_content = script_content.replace(f'$d_ff', df)
                script_content = script_content.replace(f'$e_layers', el)
                script_content = script_content.replace(f'$train_epochs', epochs)
                script_content = script_content.replace(f'$loss', loss)
                script_content = script_content.replace(f'$learning_rate', lr)
                script_content = script_content.replace(f'$lradj', lr_strategy)
                
                # 定义输出文件名
                output_file = os.path.join(output_dir, f'{file_name}.sh')
                
                # 写入新的 shell 脚本
                with open(output_file, 'w') as file:
                    file.write(script_content)

                # print(f'Generated {output_file}')

1474560
1500
delete current folder!
delete current folder!
delete current folder!
delete current folder!
589824
1500
delete current folder!
delete current folder!
delete current folder!
delete current folder!
